# 从字符到注意力，再亲手训练一个语言模型

浏览器 Python 实验：按顺序运行，变量在单元之间共享。图表来自当前代码计算。


## 1 · 建立词表和张量

字符切分只是教学 tokenizer，不是 BPE。修改 text，观察词表大小和序列长度。后续单元使用这里的变量。

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

import math, random
random.seed(7)
text = "hello model. hello world. hello model. "
vocab = sorted(set(text))
to_id = {c: i for i, c in enumerate(vocab)}
ids = [to_id[c] for c in text]
V, D = len(vocab), 4
embedding = [[random.uniform(-1,1) for _ in range(D)] for _ in vocab]
x = [embedding[i] for i in ids[:8]]
print("vocab:", to_id)
print("input IDs:", ids[:8])
print("embedding shape:", (len(x), D))
print("first vector:", x[0])

## 2 · 因果注意力

为简化观察，Q、K、V 都直接用 embedding；真实层各有可训练投影。第一行只能看自己。

In [ ]:
def softmax(z):
    e = [math.exp(v-max(z)) for v in z]
    return [v/sum(e) for v in e]
A, hidden = [], []
for i, q in enumerate(x):
    scores = [sum(a*b for a,b in zip(q,k))/D**0.5 for k in x[:i+1]]
    a = softmax(scores) + [0.0]*(len(x)-i-1)
    A.append(a)
    hidden.append([sum(a[j]*x[j][d] for j in range(len(x))) for d in range(D)])
print("attention shape:", (len(A),len(A)))
print("first row:", A[0])
print("hidden shape:", (len(hidden), D))
assert A[0] == [1.0]+[0.0]*(len(x)-1)
display_plot(list(range(len(A))), A[-1], "Last query attention", "key position", "weight")

## 3 · 真实梯度训练

现在训练独立的 bigram 模型：仅凭当前字符预测下一个字符。它不使用上一格 attention，也不是 Transformer；这样可以完整看清交叉熵梯度。

In [ ]:
W = [[0.0]*V for _ in range(V)]
pairs = list(zip(ids[:-1], ids[1:]))
learning_rate, epochs = 4.0, 120
losses = []
for epoch in range(epochs):
    grad = [[0.0]*V for _ in range(V)]
    loss = 0.0
    for source, target in pairs:
        p = softmax(W[source])
        loss -= math.log(max(p[target],1e-12))/len(pairs)
        for j in range(V):
            grad[source][j] += (p[j]-(j==target))/len(pairs)
    for i in range(V):
        for j in range(V):
            W[i][j] -= learning_rate*grad[i][j]
    losses.append(loss)
print("parameters:", V*V)
print("initial / final training NLL:", losses[0], losses[-1])
print("This measures training fit, not held-out generalization.")
display_plot(list(range(epochs)), losses, "Training cross-entropy", "epoch", "NLL")

## 4 · 输入输出与采样

修改 temperature 对比概率；模型只记住相邻字符的统计关系，不会理解世界。采样种子固定，便于比较。

In [ ]:
temperature = 0.7
assert temperature > 0
random.seed(11)
current = to_id['h']
generated = [vocab[current]]
print("logits shape:", (1, V))
p = softmax([v/temperature for v in W[current]])
print("top next characters:", sorted(zip(vocab,p),key=lambda z:-z[1])[:5])
for _ in range(60):
    p = softmax([v/temperature for v in W[current]])
    current = random.choices(range(V), weights=p)[0]
    generated.append(vocab[current])
print("".join(generated))

## 5 · 可选：在这个 Notebook 加载真实 MiniLM

首次单独运行此格会下载约 23 MB 权重及推理库，需要访问 Hugging Face 和 jsDelivr。真实推理在本机执行；这是英文 embedding 模型，不是上面训练的 bigram。此格不包含在“运行全部”中，需要博客提供的 browser_models 接口。修改两句话，重新运行，观察真实 token、mask 与 384 维输出。

In [ ]:
import json
from browser_models import embed
texts = ["A cat is sleeping on the sofa.", "A kitten is resting on a couch."]
result = json.loads(await embed(texts))
print("actual input IDs:", result['ids'])
print("actual tokens:", result['tokens'])
print("attention masks:", result['masks'])
print("output shape:", result['dims'])
print("cosine similarity:", result['cosine'])
vector = result['embeddings'][0]
display_plot(list(range(32)), vector[:32], "Real MiniLM embedding (first 32 dimensions)", "dimension", "value")
